In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
from pathlib import Path

# Locate the project inside your Machine Learning folder.
ROOT = Path(
    r"F:\Uni projects_5th sem\Machine Learning"
    r"\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features"
)

train_df = pd.read_csv(ROOT / "data/processed/train.csv")
val_df = pd.read_csv(ROOT / "data/processed/val.csv")

# Select only the signal features, excluding IDs and the answer.
feature_cols = [
    col for col in train_df.columns
    if col.startswith(("F_", "U_", "FU_"))
]

X_train = train_df[feature_cols]
y_train = train_df["figo_label"]

X_val = val_df[feature_cols]
y_val = val_df["figo_label"]

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Target labels:", sorted(y_train.unique()))

Training data: (14540, 103)
Validation data: (3115, 103)
Target labels: [np.int64(0), np.int64(1)]


In [3]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=2,
)

model.fit(X_train, y_train)

print("XGBoost training complete!")

XGBoost training complete!


In [4]:
# Predict using the validation features.
y_pred = model.predict(X_val)

# Compare predictions with the known answers.
print("XGBoost baseline — validation results\n")

print(f"Accuracy:  {accuracy_score(y_val, y_pred):.2%}")
print(f"Precision: {precision_score(y_val, y_pred, zero_division=0):.2%}")
print(f"Recall:    {recall_score(y_val, y_pred, zero_division=0):.2%}")
print(f"F1 score:  {f1_score(y_val, y_pred, zero_division=0):.2%}")

print("\nClassification report:")
print(classification_report(
    y_val,
    y_pred,
    labels=[0, 1],
    target_names=["Normal", "Abnormal"],
    zero_division=0,
))

print("Confusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_val, y_pred, labels=[0, 1]),
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"],
))

XGBoost baseline — validation results

Accuracy:  93.10%
Precision: 93.25%
Recall:    93.97%
F1 score:  93.61%

Classification report:
              precision    recall  f1-score   support

      Normal       0.93      0.92      0.92      1439
    Abnormal       0.93      0.94      0.94      1676

    accuracy                           0.93      3115
   macro avg       0.93      0.93      0.93      3115
weighted avg       0.93      0.93      0.93      3115

Confusion matrix:
                 Predicted normal  Predicted abnormal
Actual normal                1325                 114
Actual abnormal               101                1575
